# Evaluating Regression Models Using MAE

Mean Absolute Error is one of the most intuitive regression metrics because it measures average prediction error in the same units as the target variable.

This lesson explains what MAE is, how it differs from MSE and RMSE, how to compare it against a baseline, and how to use cross-validation with MAE safely.

## Why MAE matters

MAE is easy to communicate because it answers a simple question: on average, how far off are our predictions?

It is especially useful when you want:

- An error metric in the same units as the target
- A metric that is easy for non-technical stakeholders to understand
- A less outlier-sensitive alternative to MSE or RMSE
- A clear way to compare a real model against a baseline

In [ ]:
import numpy as np
import pandas as pd

from sklearn.dummy import DummyRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import cross_val_score, train_test_split

rng = np.random.default_rng(53)
n_rows = 500

df = pd.DataFrame({
    'size_sqft': rng.integers(500, 3500, size=n_rows),
    'bedrooms': rng.integers(1, 6, size=n_rows),
    'age_years': rng.integers(1, 40, size=n_rows),
})

df['price'] = (
    df['size_sqft'] * 1100
    + df['bedrooms'] * 16000
    - df['age_years'] * 1700
    + rng.normal(0, 30000, size=n_rows)
).round(2)

df.head()

## Baseline comparison

MAE is only useful when compared to a reference point. For regression, the standard baseline is a constant predictor such as the mean target value.

The model should be evaluated on the same held-out test set as the baseline.

In [ ]:
TARGET_COLUMN = 'price'
X = df.drop(columns=[TARGET_COLUMN])
y = df[TARGET_COLUMN]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

baseline = DummyRegressor(strategy='mean')
baseline.fit(X_train, y_train)
baseline_preds = baseline.predict(X_test)

model = LinearRegression()
model.fit(X_train, y_train)
model_preds = model.predict(X_test)

baseline_mae = mean_absolute_error(y_test, baseline_preds)
model_mae = mean_absolute_error(y_test, model_preds)
improvement = baseline_mae - model_mae
pct_improve = (improvement / baseline_mae) * 100

print(f'Baseline MAE: {baseline_mae:.2f}')
print(f'Model MAE:    {model_mae:.2f}')
print(f'Improvement:  {improvement:.2f} ({pct_improve:.1f}%)')

## Cross-validation with MAE

Cross-validation gives a more stable view of performance than a single test split. Scikit-learn reports negative MAE for scoring, so the sign must be flipped back before reporting.

In [ ]:
cv_scores = cross_val_score(model, X_train, y_train, cv=5, scoring='neg_mean_absolute_error')
mae_scores = -cv_scores

print(f'CV MAE per fold: {np.round(mae_scores, 2)}')
print(f'Mean CV MAE:     {mae_scores.mean():.3f}')
print(f'Std CV MAE:      {mae_scores.std():.3f}')

metrics = {
    'baseline_rmse': np.sqrt(mean_squared_error(y_test, baseline_preds)),
    'baseline_mae': baseline_mae,
    'baseline_r2': r2_score(y_test, baseline_preds),
    'model_rmse': np.sqrt(mean_squared_error(y_test, model_preds)),
    'model_mae': model_mae,
    'model_r2': r2_score(y_test, model_preds),
}

print({k: round(v, 3) for k, v in metrics.items()})

## Practical checklist

- Compute MAE on held-out data only.
- Compare the model against a mean or median baseline.
- Interpret MAE in the target's original units.
- Report MAE alongside RMSE and R².
- Use cross-validation to check stability across folds.

## Bonus resources

- Scikit-learn Model Evaluation Guide
- Understanding Regression Metrics - Scikit-learn